In [6]:
# Kerakli kutubxonalarni yuklash
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [7]:
# Datasetni o'qish
data = pd.read_csv('titanic.csv')

In [8]:
# Ma'lumotlar o'lchami va umumiy ma'lumotlar (EDA)
print(data.shape)
data.info()

(891, 12)
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [9]:
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [10]:
# Asosiy statistik ko'rsatkichlar
data.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [11]:
# Omon qolganlar va sinflar bo'yicha taqsimot (indekslar to'g'rilandi: 0 - halok bo'lgan, 1 - omon qolgan)
all_survived = data.Survived.value_counts(normalize=True)
print(f"Omon qolmaganlar: {all_survived[0]*100:.2f} %")
print(f"Tiriklar: {all_survived[1]*100:.2f} %")
print(f"Omonlar soni: {data.Survived.value_counts()[1]}")
print(data.Pclass.value_counts())

Omon qolmaganlar: 61.62 %
Tiriklar: 38.38 %
Omonlar soni: 342
Pclass
3    491
1    216
2    184
Name: count, dtype: int64


In [12]:
# Jins va Pclass bo'yicha omon qolish ehtimolligi
print(data.groupby('Sex')["Survived"].mean())
print(data.groupby('Pclass')["Survived"].mean())

Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64
Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64


In [13]:
# 12 yoshgacha bolalarning omon qolish ulushi
data.groupby(data['Age'] < 12)['Survived'].mean()

Age
False    0.368165
True     0.573529
Name: Survived, dtype: float64

In [14]:
# Hamroh bilan kelgan yo'lovchilarning omon qolish ulushi (mantiqiy qavslar to'g'rilandi)
data.groupby((data['SibSp'] > 0) | (data['Parch'] > 0))['Survived'].mean()

False    0.303538
True     0.505650
Name: Survived, dtype: float64

In [15]:
# Jins va Pclass kesishmasida omon qolish matritsasi
pd.crosstab(data['Sex'], data['Pclass'], values=data['Survived'], aggfunc='mean')

Pclass,1,2,3
Sex,,,
female,0.968085,0.921053,0.500000
male,0.368852,0.157407,0.135447


In [16]:
# Korrelyatsiyalar tahlili
print(data[['Pclass', 'Fare']].corr())
print(data[['Pclass', 'Survived']].corr())

        Pclass    Fare
Pclass  1.0000 -0.5495
Fare   -0.5495  1.0000
            Pclass  Survived
Pclass    1.000000 -0.338481
Survived -0.338481  1.000000


In [17]:
# Yetishmayotgan qiymatlar foizi (len(df) o'rniga to'g'ri data hajmi olindi)
(data.isnull().sum() / len(data) * 100).round(1)

PassengerId     0.0
Survived        0.0
Pclass          0.0
Name            0.0
Sex             0.0
Age            19.9
SibSp           0.0
Parch           0.0
Ticket          0.0
Fare            0.0
Cabin          77.1
Embarked        0.2
dtype: float64

In [18]:
# Bo'sh qiymatlarni tozalash va to'ldirish
data = data.drop('Cabin', axis=1)                                # 77% bo'sh bo'lgan ustunni o'chirish
data['Age'] = data['Age'].fillna(data['Age'].median())           # Yoshni median bilan to'ldirish
data['Embarked'] = data['Embarked'].fillna(data['Embarked'].mode()[0])  # Embarked dagi 2 ta bo'sh qiymatni moda bilan to'ldirish

# Tozalashdan keyingi holat
(data.isnull().sum() / len(data) * 100).round(1)

PassengerId    0.0
Survived       0.0
Pclass         0.0
Name           0.0
Sex            0.0
Age            0.0
SibSp          0.0
Parch          0.0
Ticket         0.0
Fare           0.0
Embarked       0.0
dtype: float64

In [19]:
# Fare ustunidagi keskin og'ishlar (Outliers) ni aniqlash
Q1, Q3 = data["Fare"].quantile([0.25, 0.75])
IQR = Q3 - Q1
out = data[data["Fare"] > (Q3 + 1.5 * IQR)]
print(f"Outlierlar soni: {len(out)}")

Outlierlar soni: 116


In [20]:
# Feature Engineering: Oila hajmi va yolg'iz sayohat qilganlik belgisi (xato to'g'rilandi)
data['family_size'] = data['SibSp'] + data['Parch'] + 1
data['is_alone'] = (data['family_size'] == 1).astype(int)

print(data['family_size'].describe())
print(data['is_alone'].value_counts())

count    891.000000
mean       1.904602
std        1.613459
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max       11.000000
Name: family_size, dtype: float64
is_alone
1    537
0    354
Name: count, dtype: int64


In [21]:
# Feature Engineering: Ismlardan unvonlarni ajratish (Mr, Mrs, Miss, Master...)
data['Title'] = data['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
data['Title'] = data['Title'].replace(['Lady', 'Countess','Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
data['Title'] = data['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

# Fare anomaliyalarini yumshatish uchun log-transformatsiya
data['Fare_Log'] = np.log1p(data['Fare'])

<>:2: SyntaxWarning: invalid escape sequence '\.'
<>:2: SyntaxWarning: invalid escape sequence '\.'
C:\Users\Acer\AppData\Local\Temp\ipykernel_16100\243569262.py:2: SyntaxWarning: invalid escape sequence '\.'
  data['Title'] = data['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)


In [22]:
# Encoding (Kategorik belgilarni raqamga o'tkazish)
le = LabelEncoder()
data['Sex_encoded'] = le.fit_transform(data['Sex'])

# One-Hot Encoding (drop_first=True bilan multicollinearity oldi olinadi)
data = pd.get_dummies(data, columns=['Embarked', 'Title'], drop_first=True, dtype=int)

In [23]:
# Scaling: Raqamli ustunlarni standartlashtirish
scaler = StandardScaler()
data[['Age_scaled', 'Fare_scaled']] = scaler.fit_transform(data[['Age', 'Fare_Log']])

# Modelga kiritilmaydigan ortiqcha ustunlarni olib tashlash
drop_cols = ['PassengerId', 'Name', 'Sex', 'Ticket', 'Age', 'Fare', 'Fare_Log']
data_clean = data.drop(columns=drop_cols)

print(data_clean.info())
data_clean.head()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Survived     891 non-null    int64  
 1   Pclass       891 non-null    int64  
 2   SibSp        891 non-null    int64  
 3   Parch        891 non-null    int64  
 4   family_size  891 non-null    int64  
 5   is_alone     891 non-null    int64  
 6   Sex_encoded  891 non-null    int64  
 7   Embarked_Q   891 non-null    int64  
 8   Embarked_S   891 non-null    int64  
 9   Title_Miss   891 non-null    int64  
 10  Title_Mr     891 non-null    int64  
 11  Title_Mrs    891 non-null    int64  
 12  Title_Rare   891 non-null    int64  
 13  Age_scaled   891 non-null    float64
 14  Fare_scaled  891 non-null    float64
dtypes: float64(2), int64(13)
memory usage: 104.5 KB
None


,Survived,Pclass,SibSp,Parch,family_size,is_alone,Sex_encoded,Embarked_Q,Embarked_S,Title_Miss,Title_Mr,Title_Mrs,Title_Rare,Age_scaled,Fare_scaled
0,0,3,1,0,2,0,1,0,1,0,1,0,0,-0.565736,-0.879741
1,1,1,1,0,2,0,0,0,0,0,0,1,0,0.663861,1.361220
2,1,3,0,0,1,1,0,0,1,1,0,0,0,-0.258337,-0.798540
3,1,1,1,0,2,0,0,0,1,0,0,1,0,0.433312,1.062038
4,0,3,0,0,1,1,1,0,1,0,1,0,0,0.433312,-0.784179
